# Notebook 01 v2 — NSL-KDD Data Exploration and Preprocessing

**Methodology change vs Notebook 01 v1:**

1. **M1 fix**: one-hot encoder now fit on training data only. Test-only category values produce all-zero columns (out-of-vocabulary), not a wider schema fit on train+test union. Removes the mild data leak of using the test set's categorical schema during encoding.

2. **M6 fix**: training data split 80/20 into train + calibration portions (stratified by 5-class label). Models will be trained on the 80% train portion; calibrators will be fit on the 20% calibration portion; evaluation on the untouched official test set. Removes the methodological weakness of fitting calibrators on a split of the test set.

**Files written to `data/processed/nsl_kdd/`** (additions vs v1 marked NEW):

| File | Shape | Notes |
|---|---|---|
| `X_train.npy` | (~100,778, ~122) | 80% train slice (was 100% in v1) |
| `X_calib.npy` | (~25,195, ~122) | **NEW** — 20% calibration slice |
| `X_test.npy` | (22,544, ~122) | Untouched official test |
| `y_train_binary.npy`, `y_train_5class.npy` | (~100,778,) | Train labels |
| `y_calib_binary.npy`, `y_calib_5class.npy` | (~25,195,) | **NEW** — calibration labels |
| `y_test_binary.npy`, `y_test_5class.npy` | (22,544,) | Test labels |
| `idx_train_in_orig.npy` | (~100,778,) | **NEW** — train indices in original train set |
| `idx_calib_in_orig.npy` | (~25,195,) | **NEW** — calibration indices in original train set |
| `oov_summary.json` | — | **NEW** — categorical OOV documentation |
| `split_methodology.json` | — | **NEW** — paper-ready split description |
| `feature_names.json`, `class_mappings.json`, `scaler.pkl` | — | unchanged from v1 |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

RAW_DIR = Path(REPO) / 'data' / 'raw' / 'nsl_kdd'
OUT_DIR = Path(REPO) / 'data' / 'processed' / 'nsl_kdd_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data dir:       {RAW_DIR}')
print(f'Processed out dir:  {OUT_DIR}')
print(f'Files in raw dir:   {sorted(p.name for p in RAW_DIR.iterdir())}')

## 2. Load raw data

In [ ]:
NSL_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

CATEGORICAL = ['protocol_type', 'service', 'flag']

train_raw = pd.read_csv(RAW_DIR / 'KDDTrain+.txt', names=NSL_COLUMNS)
test_raw  = pd.read_csv(RAW_DIR / 'KDDTest+.txt',  names=NSL_COLUMNS)

train_raw = train_raw.drop(columns=['difficulty'])
test_raw  = test_raw.drop(columns=['difficulty'])

print(f'Train: {train_raw.shape}')
print(f'Test:  {test_raw.shape}')

## 3. Attack-category mapping (same as v1)

In [ ]:
ATTACK_TO_CATEGORY = {
    'normal': 'Normal',
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS',
    'processtable': 'DoS', 'udpstorm': 'DoS', 'worm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R',
}

train_raw['category'] = train_raw['label'].map(ATTACK_TO_CATEGORY)
test_raw['category']  = test_raw['label'].map(ATTACK_TO_CATEGORY)

unmapped_train = train_raw[train_raw['category'].isnull()]['label'].unique()
unmapped_test  = test_raw[test_raw['category'].isnull()]['label'].unique()
if len(unmapped_train) or len(unmapped_test):
    print(f'⚠ Unmapped labels:\n  train: {unmapped_train}\n  test:  {unmapped_test}')
else:
    print('✓ All attack labels mapped to categories')

print(f'\nCategory distribution in train:')
print(train_raw['category'].value_counts())
print(f'\nCategory distribution in test:')
print(test_raw['category'].value_counts())

## 4. One-hot encoding — M1 FIX (fit on train only)

The v1 notebook concatenated train and test before `pd.get_dummies`, which means the encoder schema implicitly saw test-set categorical values. v2 fits the encoder schema on train only. Categories that appear only in test produce all-zero one-hot rows for those columns (out-of-vocabulary handling).

In [ ]:
train_X_df = train_raw.drop(columns=['label', 'category'])
test_X_df  = test_raw.drop(columns=['label', 'category'])

# Document the out-of-vocabulary impact for the paper
oov_summary = []
for col in CATEGORICAL:
    train_vals = set(train_X_df[col].unique())
    test_vals  = set(test_X_df[col].unique())
    oov = test_vals - train_vals
    oov_count = int(test_X_df[col].isin(oov).sum())
    oov_summary.append({
        'column': col,
        'train_unique': len(train_vals),
        'test_unique': len(test_vals),
        'test_only_values': sorted(oov),
        'test_rows_affected': oov_count,
    })
    print(f'{col}: train={len(train_vals)} unique, test={len(test_vals)} unique')
    if oov:
        print(f'  Test-only values to be encoded as zero: {sorted(oov)}')
        print(f'  Test rows affected: {oov_count}')

# One-hot encode TRAIN ONLY — this defines the canonical schema
train_X_encoded = pd.get_dummies(train_X_df, columns=CATEGORICAL, prefix=CATEGORICAL, dtype=np.float32)
feature_names = list(train_X_encoded.columns)

# One-hot encode TEST using the same schema. Test-only categories disappear; missing columns filled with 0.
test_X_encoded_raw = pd.get_dummies(test_X_df, columns=CATEGORICAL, prefix=CATEGORICAL, dtype=np.float32)
test_X_encoded = test_X_encoded_raw.reindex(columns=feature_names, fill_value=0.0)

X_train_df = train_X_encoded.reset_index(drop=True)
X_test_df  = test_X_encoded.reset_index(drop=True)

print(f'\nAfter one-hot encoding (fit on train only):')
print(f'  X_train shape: {X_train_df.shape}')
print(f'  X_test shape:  {X_test_df.shape}')
print(f'  Feature count: {len(feature_names)}')
print(f'\nFirst 5 feature names: {feature_names[:5]}')
print(f'Last 5 feature names:  {feature_names[-5:]}')

with open(OUT_DIR / 'oov_summary.json', 'w') as f:
    json.dump(oov_summary, f, indent=2, default=str)

## 5. Standardisation (StandardScaler fit on train only — unchanged from v1)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.values).astype(np.float32)
X_test  = scaler.transform(X_test_df.values).astype(np.float32)

print(f'After standardisation:')
print(f'  X_train: shape={X_train.shape}, mean={X_train.mean():.4f}, std={X_train.std():.4f}')
print(f'  X_test:  shape={X_test.shape},  mean={X_test.mean():.4f},  std={X_test.std():.4f}')

## 6. Label encoding (unchanged from v1)

In [ ]:
y_train_binary = (train_raw['category'] != 'Normal').astype(np.int64).values
y_test_binary  = (test_raw['category']  != 'Normal').astype(np.int64).values

CATEGORY_TO_INT = {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}
INT_TO_CATEGORY = {v: k for k, v in CATEGORY_TO_INT.items()}

y_train_5class = train_raw['category'].map(CATEGORY_TO_INT).astype(np.int64).values
y_test_5class  = test_raw['category'].map(CATEGORY_TO_INT).astype(np.int64).values

print(f'Binary: train Normal={np.sum(y_train_binary==0):,}, Attack={np.sum(y_train_binary==1):,}')
print(f'         test Normal={np.sum(y_test_binary==0):,}, Attack={np.sum(y_test_binary==1):,}')
print(f'\n5-class (counts per class):')
for i in range(5):
    print(f'  {i} = {INT_TO_CATEGORY[i]:<8}  train: {np.sum(y_train_5class==i):>7,}   test: {np.sum(y_test_5class==i):>5,}')

## 7. Train/calibration split — M6 FIX (calibrator no longer fit on test)

v1 used a 50/50 split of the *test* set for calibration vs evaluation. v2 instead carves a 20% calibration set out of the *training* data (stratified by 5-class label to preserve U2R proportions). Models will be trained on the 80% train slice, calibrators fit on the 20% calibration slice, evaluation on the untouched official test set.

In [ ]:
idx_all = np.arange(len(X_train))
idx_train_new, idx_calib = train_test_split(
    idx_all,
    test_size=0.20,
    stratify=y_train_5class,
    random_state=SEED,
)

X_train_new = X_train[idx_train_new]
X_calib     = X_train[idx_calib]

y_train_binary_new = y_train_binary[idx_train_new]
y_calib_binary     = y_train_binary[idx_calib]

y_train_5class_new = y_train_5class[idx_train_new]
y_calib_5class     = y_train_5class[idx_calib]

print(f'Train/calibration split (stratified by 5-class):')
print(f'  Train: {len(idx_train_new):,} samples ({len(idx_train_new)/len(X_train)*100:.1f}%)')
print(f'  Calib: {len(idx_calib):,} samples ({len(idx_calib)/len(X_train)*100:.1f}%)')

print(f'\nPer-class distribution after split:')
print(f'  {"Class":<10}  {"Train":>10}  {"Calib":>10}  {"Test":>10}')
for c in range(5):
    n_train = int(np.sum(y_train_5class_new == c))
    n_calib = int(np.sum(y_calib_5class == c))
    n_test  = int(np.sum(y_test_5class == c))
    print(f'  {INT_TO_CATEGORY[c]:<10}  {n_train:>10,}  {n_calib:>10,}  {n_test:>10,}')

# Overwrite the legacy names so downstream cells use the new split
X_train = X_train_new
y_train_binary = y_train_binary_new
y_train_5class = y_train_5class_new

## 8. Class-distribution figure

In [ ]:
FIG_DIR = Path(REPO) / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary distribution: train / calib / test side by side
x = np.arange(2)
for offset, vals, label, colour in [
    (-0.27, [np.sum(y_train_binary==0), np.sum(y_train_binary==1)], 'Train', '#4C72B0'),
    (0.00,  [np.sum(y_calib_binary==0), np.sum(y_calib_binary==1)], 'Calib', '#55A868'),
    (0.27,  [np.sum(y_test_binary==0),  np.sum(y_test_binary==1)],  'Test',  '#DD8452'),
]:
    axes[0].bar(x + offset, vals, width=0.27, label=label, color=colour)
axes[0].set_xticks(x); axes[0].set_xticklabels(['Normal', 'Attack'])
axes[0].set_title('NSL-KDD v2 — binary distribution')
axes[0].set_ylabel('Number of samples')
axes[0].legend()

# 5-class distribution (log scale)
x = np.arange(5)
names = [INT_TO_CATEGORY[i] for i in range(5)]
for offset, ys, label, colour in [
    (-0.27, y_train_5class, 'Train', '#4C72B0'),
    (0.00,  y_calib_5class, 'Calib', '#55A868'),
    (0.27,  y_test_5class,  'Test',  '#DD8452'),
]:
    vals = [int(np.sum(ys == i)) for i in range(5)]
    axes[1].bar(x + offset, vals, width=0.27, label=label, color=colour)
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_title('NSL-KDD v2 — 5-class distribution (log)')
axes[1].set_ylabel('Samples (log)')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'nslkdd_v2_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nFigure saved: {FIG_DIR / "nslkdd_v2_class_distribution.png"}')

## 9. Save processed files

In [ ]:
# Feature arrays
np.save(OUT_DIR / 'X_train.npy',  X_train)
np.save(OUT_DIR / 'X_calib.npy',  X_calib)
np.save(OUT_DIR / 'X_test.npy',   X_test)

# Binary labels
np.save(OUT_DIR / 'y_train_binary.npy',  y_train_binary)
np.save(OUT_DIR / 'y_calib_binary.npy',  y_calib_binary)
np.save(OUT_DIR / 'y_test_binary.npy',   y_test_binary)

# 5-class labels
np.save(OUT_DIR / 'y_train_5class.npy',  y_train_5class)
np.save(OUT_DIR / 'y_calib_5class.npy',  y_calib_5class)
np.save(OUT_DIR / 'y_test_5class.npy',   y_test_5class)

# Indices into the original training set (for auditability)
np.save(OUT_DIR / 'idx_train_in_orig.npy', idx_train_new)
np.save(OUT_DIR / 'idx_calib_in_orig.npy', idx_calib)

# Feature names
with open(OUT_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

# Class mappings
class_info = {
    'binary': {'0': 'Normal', '1': 'Attack'},
    'multiclass_5': INT_TO_CATEGORY,
    'attack_to_category': ATTACK_TO_CATEGORY,
}
with open(OUT_DIR / 'class_mappings.json', 'w') as f:
    json.dump(class_info, f, indent=2)

# Scaler
with open(OUT_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Methodology summary (paper-ready)
methodology = {
    'version': 'v2',
    'design': 'train/calibration/test (three-way) split',
    'train_size': int(len(X_train)),
    'calib_size': int(len(X_calib)),
    'test_size': int(len(X_test)),
    'split_method': 'stratified by 5-class label, random_state=SEED, test_size=0.20',
    'fixes_vs_v1': [
        'M1: one-hot encoder fit on train only (was fit on train+test union in v1)',
        'M6: calibrator will be fit on a held-out portion of train, not on a split of test',
    ],
    'notes': [
        'Train portion: model fitting (Notebook 02 v2)',
        'Calibration portion: fitting per-class isotonic calibrators (Notebook 03 v2)',
        'Test portion: untouched official NSL-KDD test set, used for all evaluation',
        'Test-only categorical values produce all-zero one-hot columns (out-of-vocabulary handling)',
    ],
}
with open(OUT_DIR / 'split_methodology.json', 'w') as f:
    json.dump(methodology, f, indent=2)

print(f'✓ Saved all v2 processed files to: {OUT_DIR}\n')
for p in sorted(OUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:30s}  {size_kb:>10,.1f} KB')

## 10. Round-trip verification

In [ ]:
# Reload from disk and verify everything
X_train_r = np.load(OUT_DIR / 'X_train.npy')
X_calib_r = np.load(OUT_DIR / 'X_calib.npy')
X_test_r  = np.load(OUT_DIR / 'X_test.npy')

assert X_train_r.shape == X_train.shape
assert X_calib_r.shape == X_calib.shape
assert X_test_r.shape  == X_test.shape

# Disjointness: train and calibration indices must not overlap
idx_train_check = np.load(OUT_DIR / 'idx_train_in_orig.npy')
idx_calib_check = np.load(OUT_DIR / 'idx_calib_in_orig.npy')
overlap = set(idx_train_check) & set(idx_calib_check)
assert len(overlap) == 0, f'Train/calib overlap: {len(overlap)} samples leak'

# Coverage: train + calib must cover the original training set
assert len(idx_train_check) + len(idx_calib_check) == 125973, 'Train+calib != original train size'

print('✓ All files round-trip correctly.')
print('✓ Train and calibration sets are disjoint.')
print('✓ Train + calibration covers the original training set exactly.')
print()
print(f'Shapes:')
print(f'  X_train: {X_train_r.shape}  (80% of original train)')
print(f'  X_calib: {X_calib_r.shape}  (20% of original train)')
print(f'  X_test:  {X_test_r.shape}   (untouched official test)')

## 11. Commit and push

In [ ]:
os.chdir(REPO)
!git add notebooks/01_data_exploration_v2.ipynb
!git add data/processed/nsl_kdd_v2/
!git add results/figures/nslkdd_v2_class_distribution.png
!git status --short
!git commit -m 'Notebook 01 v2: NSL-KDD preprocessing with M1 + M6 fixes (encoder on train only; calibration split from train, not test)'
!git push origin main